# set up

In [1]:
from google.colab import drive
drive.mount('/content/drive')
dic = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/01OrginalData/')
data = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/02DataForAnalysis/')
result = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/03Results/')
brief = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/04DataForR/')

Mounted at /content/drive


In [2]:
#set up
import numpy as np
import pandas as pd
import spacy

# load the data

In [3]:
ad_clinical = pd.read_csv( result + 'ad_clinical_lexicon_reduced.csv',engine='python')
ad_clinical

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm,['mhm'],1,['mhm'],['INTJ'],['UH'],NaN,NaN,NaN,NaN
1,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a,['a'],1,['a'],['PRON'],['DT'],NaN,NaN,NaN,NaN
2,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young,['young'],1,['young'],['ADJ'],['JJ'],6.31,4.09,5.60,2.00
3,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,boy,['boy'],1,['boy'],['PROPN'],['NNP'],5.84,4.11,5.50,1.61
4,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,going,['going'],1,['go'],['VERB'],['VBG'],NaN,NaN,NaN,2.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7103,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,the,['the'],1,['the'],['PRON'],['DT'],NaN,NaN,NaN,2.37
7104,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,other,['other'],1,['other'],['ADJ'],['JJ'],5.41,3.48,6.00,2.37
7105,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,cabinet,['cabinet'],1,['cabinet'],['NOUN'],['NN'],5.10,3.75,5.78,1.47
7106,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,with,['with'],1,['with'],['ADP'],['IN'],NaN,NaN,NaN,2.37


# keep the id and idx columns along with the unique clean_text values

In [4]:
df = ad_clinical[["idx","id","clean_text"]].drop_duplicates().reset_index(drop=True) #.drop_duplicates():Removes duplicates by comparing all three columns.
df

,idx,id,clean_text
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...
1,1,dementia-003-0,here's a cookie jar . and the lid is off the c...
2,2,dementia-005-2,okay he's falling off a chair . she's running ...
3,3,dementia-007-3,well the girl is telling the boy to get the co...
4,4,dementia-010-0,oh boy . wowie the boy's going up on a cookiej...
...,...,...,...
80,80,dementia-369-0,mhm . well she's spilling the water from washi...
81,81,dementia-381-1,well the boy is in the cookie jar . and his si...
82,82,dementia-450-1,. well a little boy is stealing cookies out of...
83,83,dementia-458-0,oh . huh ? which moves ? oh oh . the boy is ta...


In [5]:
import re
def attach_punctuation(text):
    # Use regex to remove space before punctuation
    return re.sub(r'\s+([?.!,:;])', r'\1', text)

df['clean_text'] = df['clean_text'].apply(attach_punctuation)
df

,idx,id,clean_text
0,0,dementia-001-2,mhm. there's a young boy going in a cookie jar...
1,1,dementia-003-0,here's a cookie jar. and the lid is off the co...
2,2,dementia-005-2,okay he's falling off a chair. she's running t...
3,3,dementia-007-3,well the girl is telling the boy to get the co...
4,4,dementia-010-0,oh boy. wowie the boy's going up on a cookieja...
...,...,...,...
80,80,dementia-369-0,mhm. well she's spilling the water from washin...
81,81,dementia-381-1,well the boy is in the cookie jar. and his sis...
82,82,dementia-450-1,. well a little boy is stealing cookies out of...
83,83,dementia-458-0,oh. huh? which moves? oh oh. the boy is taking...


# VADER sentiment analysis

In [17]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.2 MB/s eta 0:00:00


In [18]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

In [19]:
# Define a function to get sentiment scores and classify sentiment
def analyze_VADER_sentiment(text):
    # Check if the input is a string; if not, convert it to a string.
    if not isinstance(text, str):
        text = str(text)

    scores = analyzer.polarity_scores(text)

    # Classify sentiment based on the compound score
    if scores['compound'] >= 0.05:
        sentiment = 'Positive'
    elif scores['compound'] <= -0.05:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'

    return pd.Series([scores['neg'], scores['neu'], scores['pos'], scores['compound'], sentiment])


In [20]:
# Apply the sentiment analysis function to the 'clean_text' column
df[['VADER_neg', 'VADER_neu', 'VADER_pos', 'VADER_compound', 'VADER_sentiment']] = df['clean_text'].apply(analyze_VADER_sentiment)
df

,idx,id,clean_text,average_similarity,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,0,dementia-001-2,mhm. there's a young boy going in a cookie jar...,0.256047,0.032,0.946,0.022,-0.0258,Neutral
1,1,dementia-003-0,here's a cookie jar. and the lid is off the co...,0.249428,0.014,0.908,0.078,0.9161,Positive
2,2,dementia-005-2,okay he's falling off a chair. she's running t...,0.174167,0.063,0.787,0.150,0.2960,Positive
3,3,dementia-007-3,well the girl is telling the boy to get the co...,0.372560,0.058,0.884,0.058,-0.0129,Neutral
4,4,dementia-010-0,oh boy. wowie the boy's going up on a cookieja...,0.204574,0.007,0.944,0.049,0.8146,Positive
...,...,...,...,...,...,...,...,...,...
80,80,dementia-369-0,mhm. well she's spilling the water from washin...,0.156045,0.000,0.941,0.059,0.5106,Positive
81,81,dementia-381-1,well the boy is in the cookie jar. and his sis...,0.236343,0.000,0.896,0.104,0.7783,Positive
82,82,dementia-450-1,. well a little boy is stealing cookies out of...,0.319051,0.075,0.865,0.060,-0.2815,Negative
83,83,dementia-458-0,oh. huh? which moves? oh oh. the boy is taking...,0.165669,0.022,0.920,0.058,0.5514,Positive


# lexical complexity (TTR)

In [21]:
# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

In [22]:
def spacy_TTR(text):

    if not isinstance(text, str):
        return pd.Series([0, 0, 0, 0.0])  # Handle non-string inputs gracefully

    # Process the text using spaCy
    doc = nlp(text)

    # Calculate sentence count
    sent_count = len(list(doc.sents)) #doc.sents:SpaCy uses its sentence boundary detection (SBD) algorithm to split the text into sentences.

    # Extract words (excluding punctuation and spaces)
    words = [token.text.lower() for token in doc if token.is_alpha]
    word_count = len(words)

    # Calculate unique word count
    unique_word_count = len(set(words))

    # Calculate Type-Token Ratio (TTR)
    TTR = unique_word_count / word_count if word_count > 0 else 0

    # Return the results as a pandas Series
    return pd.Series([sent_count, word_count, unique_word_count, TTR])

In [23]:
# Apply the compute_TTR function to the 'clean_text' column and add new columns to the DataFrame
df[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = df['clean_text'].apply(spacy_TTR)
df

,idx,id,clean_text,average_similarity,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR
0,0,dementia-001-2,mhm. there's a young boy going in a cookie jar...,0.256047,0.032,0.946,0.022,-0.0258,Neutral,12.0,110.0,59.0,0.536364
1,1,dementia-003-0,here's a cookie jar. and the lid is off the co...,0.249428,0.014,0.908,0.078,0.9161,Positive,20.0,189.0,87.0,0.460317
2,2,dementia-005-2,okay he's falling off a chair. she's running t...,0.174167,0.063,0.787,0.150,0.2960,Positive,7.0,23.0,18.0,0.782609
3,3,dementia-007-3,well the girl is telling the boy to get the co...,0.372560,0.058,0.884,0.058,-0.0129,Neutral,9.0,87.0,54.0,0.620690
4,4,dementia-010-0,oh boy. wowie the boy's going up on a cookieja...,0.204574,0.007,0.944,0.049,0.8146,Positive,19.0,220.0,119.0,0.540909
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,80,dementia-369-0,mhm. well she's spilling the water from washin...,0.156045,0.000,0.941,0.059,0.5106,Positive,9.0,71.0,51.0,0.718310
81,81,dementia-381-1,well the boy is in the cookie jar. and his sis...,0.236343,0.000,0.896,0.104,0.7783,Positive,8.0,70.0,48.0,0.685714
82,82,dementia-450-1,. well a little boy is stealing cookies out of...,0.319051,0.075,0.865,0.060,-0.2815,Negative,10.0,69.0,45.0,0.652174
83,83,dementia-458-0,oh. huh? which moves? oh oh. the boy is taking...,0.165669,0.022,0.920,0.058,0.5514,Positive,12.0,68.0,47.0,0.691176


# Syntactic complexity

In [24]:
def syntactic_complexity_metrics(text):

    if not isinstance(text, str):
        return pd.Series([0, 0, 0, 0, 0, 0])  # Handle non-string inputs gracefully

    # Process the text using spaCy
    doc = nlp(text)

    sen_count = 0     # Total number of sentences (treated as clauses)
    total_dependents = 0
    total_coordinate_phrases = 0
    total_complex_nominals = 0

    # Iterate over sentences in the text
    for sent in doc.sents:
        sen_count += 1

        # Initialize counters for the current sentence
        sent_dependents = 0
        sent_coord_phrases = 0
        sent_complex_nominals = 0

        for token in sent:
            # 1. Count dependents per clause
            if token.dep_ in {'advcl', 'csubj', 'ccomp', 'acl', 'xcomp', 'relcl'}:
                sent_dependents += 1

            # 2. Count coordinate phrases per clause
            if token.dep_ in {'cc', 'conj'}:
                sent_coord_phrases += 1

            # 3. Count complex nominals per clause
            if token.dep_ in {"nsubj", "dobj", "pobj", "iobj", "nmod"}:
                if any(child.dep_ in {"amod", "poss", "compound"} for child in token.children):
                    sent_complex_nominals += 1

        total_dependents += sent_dependents
        total_coordinate_phrases += sent_coord_phrases
        total_complex_nominals += sent_complex_nominals

    # Avoid division by zero (if no sentences are found)
    if sen_count == 0:
        return pd.Series([0, 0, 0, 0, 0, 0])

    # Calculate per-clause averages
    dependents_per_clause = total_dependents / sen_count
    coord_phrases_per_clause = total_coordinate_phrases / sen_count
    complex_nominals_per_clause = total_complex_nominals / sen_count

    # Return results as a pandas Series
    return pd.Series([
        total_dependents, dependents_per_clause,
        total_coordinate_phrases, coord_phrases_per_clause,
        total_complex_nominals, complex_nominals_per_clause
    ])

In [25]:
df[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = df['clean_text'].apply(syntactic_complexity_metrics)
df

,idx,id,clean_text,average_similarity,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,0,dementia-001-2,mhm. there's a young boy going in a cookie jar...,0.256047,0.032,0.946,0.022,-0.0258,Neutral,12.0,110.0,59.0,0.536364,11.0,0.916667,9.0,0.750000,3.0,0.250000
1,1,dementia-003-0,here's a cookie jar. and the lid is off the co...,0.249428,0.014,0.908,0.078,0.9161,Positive,20.0,189.0,87.0,0.460317,16.0,0.800000,20.0,1.000000,3.0,0.150000
2,2,dementia-005-2,okay he's falling off a chair. she's running t...,0.174167,0.063,0.787,0.150,0.2960,Positive,7.0,23.0,18.0,0.782609,0.0,0.000000,0.0,0.000000,0.0,0.000000
3,3,dementia-007-3,well the girl is telling the boy to get the co...,0.372560,0.058,0.884,0.058,-0.0129,Neutral,9.0,87.0,54.0,0.620690,4.0,0.444444,8.0,0.888889,2.0,0.222222
4,4,dementia-010-0,oh boy. wowie the boy's going up on a cookieja...,0.204574,0.007,0.944,0.049,0.8146,Positive,19.0,220.0,119.0,0.540909,19.0,1.000000,12.0,0.631579,12.0,0.631579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,80,dementia-369-0,mhm. well she's spilling the water from washin...,0.156045,0.000,0.941,0.059,0.5106,Positive,9.0,71.0,51.0,0.718310,5.0,0.555556,2.0,0.222222,2.0,0.222222
81,81,dementia-381-1,well the boy is in the cookie jar. and his sis...,0.236343,0.000,0.896,0.104,0.7783,Positive,8.0,70.0,48.0,0.685714,3.0,0.375000,6.0,0.750000,4.0,0.500000
82,82,dementia-450-1,. well a little boy is stealing cookies out of...,0.319051,0.075,0.865,0.060,-0.2815,Negative,10.0,69.0,45.0,0.652174,3.0,0.300000,9.0,0.900000,6.0,0.600000
83,83,dementia-458-0,oh. huh? which moves? oh oh. the boy is taking...,0.165669,0.022,0.920,0.058,0.5514,Positive,12.0,68.0,47.0,0.691176,4.0,0.333333,5.0,0.416667,3.0,0.250000


# Cosine similarity

In [6]:
!pip install sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer, util

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [8]:
model = SentenceTransformer("all-mpnet-base-v2")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
data = ad_clinical[["idx","id","clean_text"]].drop_duplicates().reset_index(drop=True) #.drop_duplicates():Removes duplicates by comparing all three columns.
data

,idx,id,clean_text
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...
1,1,dementia-003-0,here's a cookie jar . and the lid is off the c...
2,2,dementia-005-2,okay he's falling off a chair . she's running ...
3,3,dementia-007-3,well the girl is telling the boy to get the co...
4,4,dementia-010-0,oh boy . wowie the boy's going up on a cookiej...
...,...,...,...
80,80,dementia-369-0,mhm . well she's spilling the water from washi...
81,81,dementia-381-1,well the boy is in the cookie jar . and his si...
82,82,dementia-450-1,. well a little boy is stealing cookies out of...
83,83,dementia-458-0,oh . huh ? which moves ? oh oh . the boy is ta...


In [10]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [11]:
# Step 1: Split text into sentences
data['clean_text'] = data['clean_text'].fillna('').astype(str)
data['sent_list'] = data['clean_text'].apply(lambda x: [sent.text.strip() for sent in nlp(x).sents])
print(data[['clean_text', 'sent_list']].head())

                                          clean_text  \
0  mhm . there's a young boy going in a cookie ja...   
1  here's a cookie jar . and the lid is off the c...   
2  okay he's falling off a chair . she's running ...   
3  well the girl is telling the boy to get the co...   
4  oh boy . wowie the boy's going up on a cookiej...   

                                           sent_list  
0  [mhm ., there's a young boy going in a cookie ...  
1  [here's a cookie jar ., and the lid is off the...  
2  [okay he's falling off a chair ., she's runnin...  
3  [well the girl is telling the boy to get the c...  
4  [oh boy ., wowie the boy's going up on a cooki...  


In [12]:
example_text = data['clean_text'].iloc[0]  # 选取第一条数据
doc = nlp(example_text)
sentences = [sent.text.strip() for sent in doc.sents]
sentences

['mhm .',
 "there's a young boy going in a cookie jar .",
 "and there's a young girl .",
 "and I'm saying he's a boy because it's hard to tell anymore .",
 "and he's in the cookie jar .",
 "and there's a stool that he is on and it already is starting to fall over .",
 'and so is the water in the sink is overflowing in the sink .',
 "I don't know about the this hickey here whether that's more than what I said .",
 'like the mother is near the girl .',
 'and she has',
 "oh I can't think of the she has the  she's trying to wipe wipe dishes .",
 'oh and stop the water from going out .']

In [13]:
# Step 2: Compute Sentence Embeddings
model = SentenceTransformer("all-mpnet-base-v2")

data['sentence_embeddings'] = data['sent_list'].apply(
    lambda sentences: [model.encode(sentence, convert_to_tensor=True) for sentence in sentences])

In [14]:
# Step 3: Compute Cosine Similarities
#Using the sentence embeddings, compute the cosine similarities between consecutive sentences for each row.

def compute_pairwise_similarity(embeddings):
    if len(embeddings) < 2:
        return []  # No pairs to compare if fewer than 2 sentences
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = util.pytorch_cos_sim(embeddings[i], embeddings[i + 1]).item()
        similarities.append(sim)
    return similarities

data['pair_similarity'] = data['sentence_embeddings'].apply(compute_pairwise_similarity)


In [15]:
# Step 4: Calculate the average similarity
data['average_similarity'] = data['pair_similarity'].apply(lambda sims: np.mean(sims) if sims else None)

In [16]:
df['average_similarity'] = data['average_similarity']
df

,idx,id,clean_text,average_similarity
0,0,dementia-001-2,mhm. there's a young boy going in a cookie jar...,0.256047
1,1,dementia-003-0,here's a cookie jar. and the lid is off the co...,0.249428
2,2,dementia-005-2,okay he's falling off a chair. she's running t...,0.174167
3,3,dementia-007-3,well the girl is telling the boy to get the co...,0.372560
4,4,dementia-010-0,oh boy. wowie the boy's going up on a cookieja...,0.204574
...,...,...,...,...
80,80,dementia-369-0,mhm. well she's spilling the water from washin...,0.156045
81,81,dementia-381-1,well the boy is in the cookie jar. and his sis...,0.236343
82,82,dementia-450-1,. well a little boy is stealing cookies out of...,0.319051
83,83,dementia-458-0,oh. huh? which moves? oh oh. the boy is taking...,0.165669


# Pos tag

In [26]:
def count_pos_tags_by_speaker(df, pos_column='sp_pos', group_column='id'):
    # Ensure the sp_pos column is in list format if it contains lists as strings
    df[pos_column] = df[pos_column].apply(lambda x: eval(x) if isinstance(x, str) else x)

    # Explode the sp_pos column to have one row per POS tag
    df_exploded = df.explode(pos_column)

    # Count occurrences of each POS tag grouped by the speaker (id)
    pos_counts = df_exploded.groupby([group_column, pos_column]).size().reset_index(name='count')

    # Pivot to create a table with POS tags as columns and counts as values
    pos_counts_pivot = pos_counts.pivot(index=group_column, columns=pos_column, values='count').fillna(0)

    # Add a new column for the total POS count by summing across all POS columns
    pos_counts_pivot['total_pos'] = pos_counts_pivot.sum(axis=1)

    return pos_counts_pivot

In [27]:
df1 = count_pos_tags_by_speaker(ad_clinical, pos_column='sp_pos', group_column='id')
df1

sp_pos,ADJ,ADP,ADV,AUX,CCONJ,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,VERB,X,total_pos
id,,,,,,,,,,,,,,,,
dementia-001-2,2.0,10.0,8.0,5.0,8.0,4.0,10.0,0.0,3.0,25.0,4.0,0.0,2.0,18.0,0.0,99.0
dementia-003-0,3.0,7.0,15.0,22.0,17.0,5.0,23.0,3.0,3.0,54.0,4.0,0.0,4.0,19.0,0.0,179.0
dementia-005-2,0.0,2.0,2.0,0.0,0.0,4.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,3.0,0.0,19.0
dementia-007-3,0.0,4.0,11.0,4.0,6.0,1.0,13.0,1.0,2.0,19.0,3.0,0.0,1.0,16.0,0.0,81.0
dementia-010-0,5.0,18.0,15.0,10.0,7.0,6.0,34.0,2.0,7.0,52.0,8.0,0.0,2.0,42.0,0.0,208.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
dementia-369-0,1.0,9.0,6.0,5.0,2.0,3.0,8.0,0.0,2.0,17.0,1.0,0.0,1.0,11.0,0.0,66.0
dementia-381-1,0.0,9.0,4.0,3.0,5.0,3.0,10.0,1.0,1.0,17.0,2.0,0.0,0.0,10.0,0.0,65.0
dementia-450-1,2.0,5.0,4.0,2.0,7.0,1.0,9.0,0.0,3.0,13.0,2.0,0.0,0.0,14.0,0.0,62.0


In [28]:
def pos_analysis(df):

    # Step 1: Create a new DataFrame with selected columns
    pos_df = df[[ 'ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos']].copy()

    # Step 2: Calculate normalized (percentage) columns for each POS type
    pos_df['ADJ%'] = (pos_df['ADJ'] / pos_df['total_pos']) * 100
    pos_df['VERB%'] = (pos_df['VERB'] / pos_df['total_pos']) * 100
    pos_df['PRON%'] = (pos_df['PRON'] / pos_df['total_pos']) * 100
    pos_df['NOUN%'] = (pos_df['NOUN'] / pos_df['total_pos']) * 100

    # Step 3: Calculate the specified POS ratios
    pos_df['Noun_to_Verb'] = pos_df['NOUN'] / (pos_df['VERB'] )
    pos_df['Pron_to_Noun'] = pos_df['PRON'] / (pos_df['NOUN'] )
    pos_df['Noun_to_Adj'] = pos_df['NOUN'] / (pos_df['ADJ'])
    pos_df['Verb_to_Adj'] = pos_df['VERB'] / (pos_df['ADJ'])

    return pos_df

In [29]:
df2 = pos_analysis(df1)
df2.head()

sp_pos,ADJ,VERB,PRON,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
id,,,,,,,,,,,,,
dementia-001-2,2.0,18.0,25.0,10.0,99.0,2.020202,18.181818,25.252525,10.101010,0.555556,2.500000,5.000000,9.000000
dementia-003-0,3.0,19.0,54.0,23.0,179.0,1.675978,10.614525,30.167598,12.849162,1.210526,2.347826,7.666667,6.333333
dementia-005-2,0.0,3.0,4.0,4.0,19.0,0.000000,15.789474,21.052632,21.052632,1.333333,1.000000,inf,inf
dementia-007-3,0.0,16.0,19.0,13.0,81.0,0.000000,19.753086,23.456790,16.049383,0.812500,1.461538,inf,inf
dementia-010-0,5.0,42.0,52.0,34.0,208.0,2.403846,20.192308,25.000000,16.346154,0.809524,1.529412,6.800000,8.400000


In [30]:
merged_df = pd.merge(df, df2[['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos',
                                            'ADJ%', 'VERB%', 'PRON%','NOUN%',
                                            'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj']],
                     on='id',
                     how='left')
merged_df

,idx,id,clean_text,average_similarity,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,0,dementia-001-2,mhm. there's a young boy going in a cookie jar...,0.256047,0.032,0.946,0.022,-0.0258,Neutral,12.0,...,10.0,99.0,2.020202,18.181818,25.252525,10.101010,0.555556,2.500000,5.000000,9.000000
1,1,dementia-003-0,here's a cookie jar. and the lid is off the co...,0.249428,0.014,0.908,0.078,0.9161,Positive,20.0,...,23.0,179.0,1.675978,10.614525,30.167598,12.849162,1.210526,2.347826,7.666667,6.333333
2,2,dementia-005-2,okay he's falling off a chair. she's running t...,0.174167,0.063,0.787,0.150,0.2960,Positive,7.0,...,4.0,19.0,0.000000,15.789474,21.052632,21.052632,1.333333,1.000000,inf,inf
3,3,dementia-007-3,well the girl is telling the boy to get the co...,0.372560,0.058,0.884,0.058,-0.0129,Neutral,9.0,...,13.0,81.0,0.000000,19.753086,23.456790,16.049383,0.812500,1.461538,inf,inf
4,4,dementia-010-0,oh boy. wowie the boy's going up on a cookieja...,0.204574,0.007,0.944,0.049,0.8146,Positive,19.0,...,34.0,208.0,2.403846,20.192308,25.000000,16.346154,0.809524,1.529412,6.800000,8.400000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,80,dementia-369-0,mhm. well she's spilling the water from washin...,0.156045,0.000,0.941,0.059,0.5106,Positive,9.0,...,8.0,66.0,1.515152,16.666667,25.757576,12.121212,0.727273,2.125000,8.000000,11.000000
81,81,dementia-381-1,well the boy is in the cookie jar. and his sis...,0.236343,0.000,0.896,0.104,0.7783,Positive,8.0,...,10.0,65.0,0.000000,15.384615,26.153846,15.384615,1.000000,1.700000,inf,inf
82,82,dementia-450-1,. well a little boy is stealing cookies out of...,0.319051,0.075,0.865,0.060,-0.2815,Negative,10.0,...,9.0,62.0,3.225806,22.580645,20.967742,14.516129,0.642857,1.444444,4.500000,7.000000
83,83,dementia-458-0,oh. huh? which moves? oh oh. the boy is taking...,0.165669,0.022,0.920,0.058,0.5514,Positive,12.0,...,9.0,64.0,0.000000,21.875000,26.562500,14.062500,0.642857,1.888889,inf,inf


# sentiment:emotion dictionary

In [31]:
ad_clinical

,idx,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,mhm,['mhm'],1,['mhm'],[INTJ],['UH'],NaN,NaN,NaN,NaN
1,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,a,['a'],1,['a'],[PRON],['DT'],NaN,NaN,NaN,NaN
2,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,young,['young'],1,['young'],[ADJ],['JJ'],6.31,4.09,5.60,2.00
3,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,boy,['boy'],1,['boy'],[PROPN],['NNP'],5.84,4.11,5.50,1.61
4,0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,going,['going'],1,['go'],[VERB],['VBG'],NaN,NaN,NaN,2.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7103,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,the,['the'],1,['the'],[PRON],['DT'],NaN,NaN,NaN,2.37
7104,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,other,['other'],1,['other'],[ADJ],['JJ'],5.41,3.48,6.00,2.37
7105,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,cabinet,['cabinet'],1,['cabinet'],[NOUN],['NN'],5.10,3.75,5.78,1.47
7106,84,dementia-461-0,cookie jar the boy and a girl and the ladder t...,with,['with'],1,['with'],[ADP],['IN'],NaN,NaN,NaN,2.37


In [32]:
dic1 = ad_clinical.groupby(['id']).agg({'Valence': lambda x: np.mean(x),
                                 'Arousal': lambda x: np.mean(x),
                                 'Dominance': lambda x: np.mean(x),
                                  'SemD': lambda x: np.mean(x)}).reset_index() #.reset_index():将 id 从索引恢复为 DataFrame 的普通列
dic1

,id,Valence,Arousal,Dominance,SemD
0,dementia-001-2,5.899231,3.999615,5.381538,2.103371
1,dementia-003-0,6.071000,3.676000,5.697200,2.155569
2,dementia-005-2,6.395000,3.863333,5.901667,2.037778
3,dementia-007-3,6.067917,3.850417,5.549167,2.074805
4,dementia-010-0,5.872742,3.706613,5.532742,2.044388
...,...,...,...,...,...
80,dementia-369-0,6.108125,3.850000,5.765625,2.113492
81,dementia-381-1,6.067647,3.822353,5.732941,2.087419
82,dementia-450-1,5.377647,4.100000,5.229412,2.080000
83,dementia-458-0,6.172857,3.882857,5.824286,2.024839


In [33]:
# save the result to the orginal csv. file
# pd.merge(): Merges the two DataFrames on the common column 'id'
# how='left': Ensures that all rows from ad_social_speakers are kept, even if there’s no matching row in pos_result.
final = pd.merge(merged_df , dic1[['id','Valence', 'Arousal', 'Dominance', 'SemD']],
                     on='id',
                     how='left')
final

,idx,id,clean_text,average_similarity,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,...,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj,Valence,Arousal,Dominance,SemD
0,0,dementia-001-2,mhm. there's a young boy going in a cookie jar...,0.256047,0.032,0.946,0.022,-0.0258,Neutral,12.0,...,25.252525,10.101010,0.555556,2.500000,5.000000,9.000000,5.899231,3.999615,5.381538,2.103371
1,1,dementia-003-0,here's a cookie jar. and the lid is off the co...,0.249428,0.014,0.908,0.078,0.9161,Positive,20.0,...,30.167598,12.849162,1.210526,2.347826,7.666667,6.333333,6.071000,3.676000,5.697200,2.155569
2,2,dementia-005-2,okay he's falling off a chair. she's running t...,0.174167,0.063,0.787,0.150,0.2960,Positive,7.0,...,21.052632,21.052632,1.333333,1.000000,inf,inf,6.395000,3.863333,5.901667,2.037778
3,3,dementia-007-3,well the girl is telling the boy to get the co...,0.372560,0.058,0.884,0.058,-0.0129,Neutral,9.0,...,23.456790,16.049383,0.812500,1.461538,inf,inf,6.067917,3.850417,5.549167,2.074805
4,4,dementia-010-0,oh boy. wowie the boy's going up on a cookieja...,0.204574,0.007,0.944,0.049,0.8146,Positive,19.0,...,25.000000,16.346154,0.809524,1.529412,6.800000,8.400000,5.872742,3.706613,5.532742,2.044388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,80,dementia-369-0,mhm. well she's spilling the water from washin...,0.156045,0.000,0.941,0.059,0.5106,Positive,9.0,...,25.757576,12.121212,0.727273,2.125000,8.000000,11.000000,6.108125,3.850000,5.765625,2.113492
81,81,dementia-381-1,well the boy is in the cookie jar. and his sis...,0.236343,0.000,0.896,0.104,0.7783,Positive,8.0,...,26.153846,15.384615,1.000000,1.700000,inf,inf,6.067647,3.822353,5.732941,2.087419
82,82,dementia-450-1,. well a little boy is stealing cookies out of...,0.319051,0.075,0.865,0.060,-0.2815,Negative,10.0,...,20.967742,14.516129,0.642857,1.444444,4.500000,7.000000,5.377647,4.100000,5.229412,2.080000
83,83,dementia-458-0,oh. huh? which moves? oh oh. the boy is taking...,0.165669,0.022,0.920,0.058,0.5514,Positive,12.0,...,26.562500,14.062500,0.642857,1.888889,inf,inf,6.172857,3.882857,5.824286,2.024839


In [34]:
final.columns

Index(['idx', 'id', 'clean_text', 'average_similarity', 'VADER_neg',
       'VADER_neu', 'VADER_pos', 'VADER_compound', 'VADER_sentiment',
       'sent_count', 'word_count', 'unique_word_count', 'TTR',
       'total_dependents', 'dependents_per_clause', 'total_coord_phrases',
       'coord_phrases_per_clause', 'total_complex_nominals',
       'complex_nominals_per_clause', 'ADJ', 'VERB', 'PRON', 'NOUN',
       'total_pos', 'ADJ%', 'VERB%', 'PRON%', 'NOUN%', 'Noun_to_Verb',
       'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj', 'Valence', 'Arousal',
       'Dominance', 'SemD'],
      dtype='object')

In [35]:
# save the results
final.to_csv(result+ 'ad_clinical_result_reduced.csv', index=False)